# FADE — IMMEDIATE retry (inline null baseline) — STANDALONE

**Per question:** solve once → correct? keep (GOOD→pool), move on. Wrong? retry the SAME question **immediately, up to twice** (blind re-sample temp 0.8, no diagnosis, no typed exemplars). A retry correct → keep, stop. Still wrong after 2 → **discard, never revisit.** No queues, no deferred phase.

Uses **`run_immediate.py`** — a standalone script that does NOT modify `kaggle_run.py`, so this is safe to run while a FADE job is using `kaggle_run.py` (even on the same repo).

**Sidebar:** GPU on · Internet on · Secrets `HF_TOKEN` + `GH_TOKEN` · repo has committed `artifacts/`.


In [1]:
# 1. Clone
import os, shutil
!pip -q install -U "transformers>=4.40" accelerate sentence-transformers sympy scikit-learn datasets scipy 2>/dev/null | tail -1
REPO='/kaggle/working/fade'
if os.path.exists(REPO): shutil.rmtree(REPO)
!git clone --depth 1 https://github.com/S2V3/fade.git {REPO}
print('cloned')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 38.2 MB/s eta 0:00:00
Cloning into '/kaggle/working/fade'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 21 (delta 0), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 356.14 KiB | 2.60 MiB/s, done.
cloned


In [2]:
# 2. VERIFY repo has the standalone runner + eval order (don't waste GPU on stale code)
import os
!cd {REPO} && test -f run_immediate.py && echo 'code OK: run_immediate.py present' || echo 'STALE - push run_immediate.py first'
!cd {REPO} && grep -q v5-hashline generation.py && echo 'generation OK' || echo 'STALE generation.py'
assert any('evalorder' in f for f in os.listdir(f'{REPO}/artifacts')), 'artifacts/ missing - prime & commit first'
print('eval order present:', os.listdir(f'{REPO}/artifacts'))


code OK: run_immediate.py present
generation OK
eval order present: ['seeds_evalorder_pool2000_test.json']


In [3]:
# 3. HF token
from kaggle_secrets import UserSecretsClient
print('HF_TOKEN length', len(UserSecretsClient().get_secret('HF_TOKEN')))


HF_TOKEN length 37


In [4]:
# 4. Config (match every arm)
MODEL='meta-llama/Llama-2-7b-chat-hf'; N=1319; BUDGET=8; TOK=320
print(MODEL,N,BUDGET,TOK)


meta-llama/Llama-2-7b-chat-hf 1319 8 320


In [5]:
import os, shutil
os.makedirs(f'{REPO}/store_immediate', exist_ok=True)
shutil.copy('/kaggle/input/datasets/ryukftw/result/results.jsonl', f'{REPO}/store_immediate/results.jsonl')
print('restored', sum(1 for _ in open(f'{REPO}/store_immediate/results.jsonl')), 'records')

restored 1160 records


In [6]:
# 5. RUN standalone immediate. Banner prints 'IMMEDIATE RUN (inline retry x2, discard-on-fail)'.
!cd {REPO} && python run_immediate.py --n-problems {N} --strategy 2 --split test \
    --model {MODEL} --exemplar-budget {BUDGET} --max-new-tokens {TOK} \
    --retries 2 --run-name immediate --show-every 50 --secret-name HF_TOKEN


Torch OK
 sentence-transformers OK
  generation module version: v5-hashline
  v5-hashline ACTIVE | N_SHOTS=8 rep_penalty=1.15 no_repeat_ngram=6
  ban_strings=['\\begin{code}', '\\end{code}', '```'] | stop_markers=['\nQuestion:', '\nQ:', '\nExample', '\nProblem:']
  preprocessing: normalize_traces=True
  model: meta-llama/Llama-2-7b-chat-hf | immediate inline retry x2
  HF token loaded from Kaggle Secret 'HF_TOKEN'
  HF identity: S2V3 (type=user)
  gated-access probe OK: meta-llama/Llama-2-7b-chat-hf reachable (16 files)
README.md: 7.93kB [00:00, 4.20MB/s]
main/train-00000-of-00001.parquet: 100%|███| 2.31M/2.31M [00:00<00:00, 2.76MB/s]
main/test-00000-of-00001.parquet: 100%|██████| 419k/419k [00:00<00:00, 1.96MB/s]
Generating test split: 100%|█████| 1319/1319 [00:00<00:00, 253530.41 examples/s]
  source: openai/gsm8k [train]
  train: 2100 problems ready (preprocessed) | dropped 0 unparseable
  source: openai/gsm8k [test]
  test: 1319 problems ready (preprocessed) | dropped 0 unparseable

In [7]:
# 6. Results
import json; s=json.load(open(f'{REPO}/store_immediate/run_summary.json'))
print('IMMEDIATE  pass1=%.2f%%  final=%.2f%%  recovered=%d  gens=%d'%(
      100*s['pass1_accuracy'],100*s['final_accuracy'],s['recovered_by_retry'],s['cost']['generations']))


IMMEDIATE  pass1=1.59%  final=3.18%  recovered=21  gens=426


In [8]:
# ===== AUTO-PUSH store_immediate/ TO GITHUB (needs Kaggle Secret GH_TOKEN) =====
import os, shutil, subprocess, datetime
from kaggle_secrets import UserSecretsClient
REPO='/kaggle/working/fade'; FOLDER='store_immediate'; BRANCH='results'
gh=UserSecretsClient().get_secret('GH_TOKEN'); AUTH=f'https://{gh}@github.com/S2V3/fade.git'
stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S'); src=f'{REPO}/{FOLDER}'
assert os.path.isdir(src), f'{src} not found'
def run(cmd,cwd):
    r=subprocess.run(cmd,cwd=cwd,capture_output=True,text=True)
    print(' '.join('***' if ('ghp_' in c or '@github' in c) else c for c in cmd)); print((r.stdout+r.stderr).strip()); return r
work=f'/kaggle/working/_push_{stamp}'
r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,work],capture_output=True,text=True)
if r.returncode!=0:
    subprocess.run(['git','clone','--depth','1',AUTH,work],check=True,capture_output=True,text=True); run(['git','checkout','-b',BRANCH],work)
shutil.copytree(src,f'{work}/results/{FOLDER}_{stamp}')
run(['git','config','user.email','fade@kaggle'],work); run(['git','config','user.name','fade'],work)
run(['git','add','-A'],work); run(['git','commit','-m',f'{FOLDER} {stamp}'],work)
push=run(['git','push','-u','origin',BRANCH],work)
if push.returncode==0 and '403' not in (push.stdout+push.stderr): print(f'\n✅ {FOLDER} pushed @ {BRANCH}/results/{FOLDER}_{stamp}/')
else: shutil.make_archive(f'/kaggle/working/{FOLDER}','zip',src); print(f'\n⚠️ push failed — wrote /kaggle/working/{FOLDER}.zip')


git checkout -b results
Switched to a new branch 'results'
git config user.email fade@kaggle

git config user.name fade

git add -A

git commit -m store_immediate 20260727_090828
[results 97f58be] store_immediate 20260727_090828
 3 files changed, 1371 insertions(+)
 create mode 100644 results/store_immediate_20260727_090828/pool.jsonl
 create mode 100644 results/store_immediate_20260727_090828/results.jsonl
 create mode 100644 results/store_immediate_20260727_090828/run_summary.json
git push -u origin results
Branch 'results' set up to track remote branch 'results' from 'origin'.
remote: 
remote: Create a pull request for 'results' on GitHub by visiting:        
remote:      https://github.com/S2V3/fade/pull/new/results        
remote: 
To https://github.com/S2V3/fade.git
 * [new branch]      results -> results

✅ store_immediate pushed @ results/results/store_immediate_20260727_090828/


**typed.final − immediate.final** = value of diagnosis vs blind inline re-sampling.
